In [ ]:
import os
def get_code_lengths(node, length=0, lengths=None):
    # Обход дерева для получения длин кодов для каждого символа
    if lengths is None:
        lengths = {}
    
    if len(node) == 2:  # Лист (символ, частота)
        # Обработка случая одного символа
        lengths[node[0]] = max(1, length)
        return lengths

    if len(node) == 4:  # Узел (None, вес, лево, право)
        get_code_lengths(node[2], length + 1, lengths)
        get_code_lengths(node[3], length + 1, lengths)
    
    return lengths

def generate_canonical_codes(lengths):
    # Сортируем символы: сначала по длине кода, затем по значению символа
    sorted_symbols = sorted(lengths.keys(), key=lambda s: (lengths[s], s))
    
    codes = {}
    code = 0
    prev_len = 0
    
    for symbol in sorted_symbols:
        curr_len = lengths[symbol]
        # Сдвигаем код на разницу длин
        code <<= (curr_len - prev_len)
        
        # Формируем битовую строку
        codes[symbol] = format(code, f'0{curr_len}b')
        
        code += 1
        prev_len = curr_len
        
    return codes

def frequency_model(b):
    d = {}
    for i in b:
        d[i] = d.get(i, 0) + 1
    return d

def huffman_encode(data, model):
    # Сортируем очередь по частоте
    queue = sorted([[char, freq] for char, freq in model.items()], key=lambda x: x[1])
    
    # Строим дерево
    while len(queue) > 1:
        one = queue.pop(0)
        two = queue.pop(0)
        # Узел: [None, вес, левый_потомок, правый_потомок]
        parent = [None, one[1] + two[1], one, two]
        
        # Вставляем обратно, сохраняя сортировку по весу
        insert_position = 0
        for i, item in enumerate(queue):
            if parent[1] > item[1]:
                insert_position = i + 1
            else:
                break
        queue.insert(insert_position, parent)
    
    tree = queue[0] if queue else None
    
    # 1. Получаем длины кодов из дерева
    lengths = get_code_lengths(tree) if tree else {}
    
    # 2. Генерируем канонические коды по длинам
    codes = generate_canonical_codes(lengths)
    
    # 3. Кодируем данные
    encoded = ''.join(codes[char] for char in data)
    
    return codes, encoded, lengths

def huffman_decode(encoded, lengths):
    if not lengths or not encoded:
        return b""
    
    # Восстанавливаем канонические коды для декодирования
    codes = generate_canonical_codes(lengths)
    reverse_codes = {v: k for k, v in codes.items()}
    
    result = []
    buffer = ""
    
    # Оптимизация: максимальная длина кода для ограничения поиска
    max_len = max(lengths.values()) if lengths else 0
    
    for bit in encoded:
        buffer += bit
        # Проверяем, есть ли такой код в таблице
        if len(buffer) <= max_len and buffer in reverse_codes:
            result.append(reverse_codes[buffer])
            buffer = ""
    
    return bytes(result)

def bits_to_bytes(bit_string):
    if not bit_string:
        return b"", 0
        
    padding = (8 - len(bit_string) % 8) % 8
    bit_string += '0' * padding
    
    result = []
    for i in range(0, len(bit_string), 8):
        byte_chunk = bit_string[i:i+8]
        result.append(int(byte_chunk, 2))
    
    return bytes(result), padding

def bytes_to_bits(data, total_bits):
    bit_string = ''.join(f'{byte:08b}' for byte in data)
    return bit_string[:total_bits]

def write_huffman_file(filename, codes, encoded_bits, lengths):
    compressed_bytes, padding = bits_to_bytes(encoded_bits)
    unique_symbols = len(codes)
    
    with open(filename, 'wb') as f:
        # 1. Количество уникальных символов
        f.write(bytes([unique_symbols]))

        # 2. Таблица: Символ + Длина кода
        for byte_val in sorted(codes.keys()):
            f.write(bytes([byte_val]))
            f.write(bytes([lengths[byte_val]]))
        
        # 3. Служебная информация
        f.write(bytes([padding]))
        f.write(len(encoded_bits).to_bytes(4, 'big'))
        
        # 4. Сжатые данные
        f.write(compressed_bytes)
    
    header_size = 1 + (unique_symbols * 2) + 1 + 4
    print(f"Записано в {filename}: {len(compressed_bytes) + header_size} байт (Заголовок: {header_size})")

def read_huffman_file(filename):
    with open(filename, 'rb') as f:
        # 1. Количество уникальных символов
        num_symbols = f.read(1)[0]
        
        # 2. Читаем длины кодов
        lengths = {}
        for _ in range(num_symbols):
            byte_val = f.read(1)[0]
            code_len = f.read(1)[0]
            lengths[byte_val] = code_len
        
        # 3. Служебная информация
        padding = f.read(1)[0]
        total_bits = int.from_bytes(f.read(4), 'big')
        
        # 4. Сжатые данные
        compressed_data = f.read()
        encoded_bits = bytes_to_bits(compressed_data, total_bits)
    
    return lengths, encoded_bits

def compress_file(input_path, output_path):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    if not data:
        with open(output_path, 'wb') as f:
            f.write(bytes([0])) # 0 символов
        return

    model = frequency_model(data)
    codes, encoded_bits, lengths = huffman_encode(data, model)
    write_huffman_file(output_path, codes, encoded_bits, lengths)
    
    print(f"Оригинал: {len(data)} байт")

def decompress_file(input_path, output_path):
    lengths, encoded_bits = read_huffman_file(input_path)
    
    if not lengths:
        with open(output_path, 'wb') as f:
            pass
        return

    result = huffman_decode(encoded_bits, lengths)
    
    with open(output_path, 'wb') as f:
        f.write(result)
    
    print(f"Распаковано: {len(result)} байт")

Оригинал: b'abracadabra'
Длины кодов: {97: 1, 114: 2, 99: 4, 100: 4, 98: 3}
Канонические коды: {97: '0', 114: '10', 98: '110', 99: '1110', 100: '1111'}
Биты: 01101001110011110110100
Восстановлено: b'abracadabra'
Совпадает: True

Записано в test_compressed.bin: 304 байт (Заголовок: 16)
Оригинал: 1100 байт
Распаковано: 1100 байт
Файлы совпадают: True
